In [1]:
#!/usr/bin/env python3
"""
Tuned MLP on ai4i‑2020 sensor data with MCC scoring
==================================================

* Early‑stopping enabled (validation_fraction = 0.15)
* Randomised search over solver, LR schedule, batch size, width/depth …
* Log‑uniform priors for alpha and learning‑rate_init
"""

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# ----------------------------------------------------------------------
# 0) Imports
# ----------------------------------------------------------------------
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    cross_val_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    matthews_corrcoef,
    make_scorer,
    classification_report,
)
from sklearn.neural_network import MLPClassifier

import scipy.stats as stats   # for log‑uniform priors

RANDOM_STATE = 42
MCC_SCORER   = make_scorer(matthews_corrcoef)

# ----------------------------------------------------------------------
# 1) Load & balance the data
# ----------------------------------------------------------------------
df = pd.read_csv("ai4i2020.csv")

FEATURE_COLS = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]
X, y = df[FEATURE_COLS], df["Machine failure"]

try:
    from imblearn.under_sampling import RandomUnderSampler
    rus   = RandomUnderSampler(sampling_strategy="auto", random_state=RANDOM_STATE)
    X_bal, y_bal = rus.fit_resample(X, y)
except ImportError:
    # simple manual undersample (class counts are 339 vs. 339 as in your grid)
    n_min       = y.value_counts().min()
    df_bal      = (
        df.groupby("Machine failure", group_keys=False)
          .apply(lambda d: d.sample(n_min, random_state=RANDOM_STATE))
          .sample(frac=1, random_state=RANDOM_STATE)   # shuffle
    )
    X_bal, y_bal = df_bal[FEATURE_COLS], df_bal["Machine failure"]

# ----------------------------------------------------------------------
# 2) Train / test split (stratified)
# ----------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_bal,
    y_bal,
    test_size=0.20,
    stratify=y_bal,
    random_state=RANDOM_STATE,
)

# ----------------------------------------------------------------------
# 3) Pre‑processing pipeline
# ----------------------------------------------------------------------
preprocessor = ColumnTransformer(
    [("scale", StandardScaler(), FEATURE_COLS)]
)

# ----------------------------------------------------------------------
# 4) Define the MLP pipeline & search space
# ----------------------------------------------------------------------
mlp_pipe = Pipeline(
    [
        ("prep", preprocessor),
        (
            "clf",
            MLPClassifier(
                max_iter=300,
                early_stopping=True,
                n_iter_no_change=10,
                validation_fraction=0.15,
                shuffle=True,
            ),
        ),
    ]
)

param_dist = {
    # network architecture
    "clf__hidden_layer_sizes": [
        (64,),
        (128,),
        (150,),
        (100, 50),
        (64, 32, 16),
    ],
    "clf__activation": ["relu", "tanh"],
    # optimisation
    "clf__solver": ["adam", "sgd"],
    "clf__learning_rate": ["constant", "adaptive"],
    "clf__learning_rate_init": stats.loguniform(1e-4, 1e-2),
    # regularisation
    "clf__alpha": stats.loguniform(1e-6, 1e-2),
    # SGD‑specific (harmless for Adam – attributes still exist)
    "clf__momentum": stats.uniform(0.8, 0.15),  # 0.80–0.95
    "clf__nesterovs_momentum": [True],
    # mini‑batch size
    "clf__batch_size": [16, 32, 64],
    # different random seeds to shake up weight initialisation
    "clf__random_state": [2, 13, 42, 99],
}

search = RandomizedSearchCV(
    mlp_pipe,
    param_distributions=param_dist,
    n_iter=60,
    scoring=MCC_SCORER,
    cv=5,
    random_state=RANDOM_STATE,
    verbose=2,
    n_jobs=-1,
)

# ----------------------------------------------------------------------
# 5) Hyper‑parameter search
# ----------------------------------------------------------------------
search.fit(X_train, y_train)

print("\n===== Best cross‑validated model =====")
print(f"CV‑best MCC : {search.best_score_:.4f}")
for k, v in search.best_params_.items():
    print(f"  {k:25s}: {v}")

best_mlp = search.best_estimator_

# ----------------------------------------------------------------------
# 6) Evaluate on the held‑out test set
# ----------------------------------------------------------------------
print("\n===== Hold‑out test performance =====")
y_pred   = best_mlp.predict(X_test)
mcc_test = matthews_corrcoef(y_test, y_pred)
print(f"Test MCC    : {mcc_test:.4f}\n")
print(classification_report(y_test, y_pred))

# ----------------------------------------------------------------------
# 7) (Optional) outer 5‑fold CV for an unbiased estimate
# ----------------------------------------------------------------------
print("===== Outer 5‑fold CV on full balanced set (optional) =====")
outer_scores = cross_val_score(
    best_mlp, X_bal, y_bal, cv=5, scoring=MCC_SCORER, n_jobs=-1
)
print("Fold MCCs :", np.round(outer_scores, 4))
print(
    f"Mean ± SD : {outer_scores.mean():.4f} ± {outer_scores.std():.4f}"
)


Fitting 5 folds for each of 60 candidates, totalling 300 fits

===== Best cross‑validated model =====
CV‑best MCC : 0.7801
  clf__activation          : relu
  clf__alpha               : 3.911539496057501e-06
  clf__batch_size          : 16
  clf__hidden_layer_sizes  : (150,)
  clf__learning_rate       : constant
  clf__learning_rate_init  : 0.008979040111598217
  clf__momentum            : 0.8616555519977347
  clf__nesterovs_momentum  : True
  clf__random_state        : 42
  clf__solver              : adam

===== Hold‑out test performance =====
Test MCC    : 0.8239

              precision    recall  f1-score   support

           0       0.92      0.90      0.91        68
           1       0.90      0.93      0.91        68

    accuracy                           0.91       136
   macro avg       0.91      0.91      0.91       136
weighted avg       0.91      0.91      0.91       136

===== Outer 5‑fold CV on full balanced set (optional) =====
Fold MCCs : [0.575  0.7941 0.8623 0.7247